# 02. Traitement des Cohortes, Binning & Transformation WoE

**Objectifs**

1. **Contrôle de représentativité des cohortes** : Vérifier par des tests statistiques (Kolmogorov-Smirnov et Chi²) que l'échantillon d'apprentissage (*Train*) et l'échantillon de validation (*Test*) présentent des distributions identiques.
2. **Discrétisation (*Optimal Binning*)** : Découper les variables continues et catégorielles en classes homogènes en respectant les contraintes Bâloises (monotonie du taux de défaut, taille minimale par classe $\ge 5\%$).
3. **Transformation WoE (*Weight of Evidence*)** : Linéariser la relation entre chaque variable explicative et le risque de défaut.
4. **Calcul de l'Information Value (IV)** : Sélectionner les variables ayant un pouvoir prédictif suffisant sur le Train.
5. **Prévention du Data Leakage** : Ajuster (*fit*) les règles de découpage et les valeurs WoE **strictement sur la cohorte Train**, puis les appliquer (*transform*) sur la cohorte Test.


In [14]:
# Importation des librairies pour le binning et le scoring
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [15]:
try:
    from optbinning import OptimalBinning
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "optbinning"])
    from optbinning import OptimalBinning


In [16]:
# Configuration des graphiques
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Chargement des cohortes (Train/test)

In [17]:
train_df = pd.read_csv("../data/sets/datatrain_sample.csv")
test_df = pd.read_csv("../data/sets/test_sample.csv")

print(f"Cohorte Apprentissage (Train) : {train_df.shape[0]} emprunteurs | Taux de défaut : {train_df['loan_status'].mean()*100:.2f}%")
print(f"Cohorte Validation (Test)     : {test_df.shape[0]} emprunteurs | Taux de défaut : {test_df['loan_status'].mean()*100:.2f}%")

Cohorte Apprentissage (Train) : 700 emprunteurs | Taux de défaut : 57.71%
Cohorte Validation (Test)     : 300 emprunteurs | Taux de défaut : 57.67%


## 2. Contrôle de Représentativité et Homogénéité des Cohortes

Avant toute transformation, nous devons valider que l'échantillon d'apprentissage et l'échantillon de validation sont issus de la même population :
- **Variables numériques** : Test de Kolmogorov-Smirnov (KS). Si $p\text{-value} > 0.05$, les distributions sont identiques.
- **Variables catégorielles** : Test du Chi² d'indépendance. Si $p\text{-value} > 0.05$, les proportions sont homogènes.


In [18]:
homogeneity_results = []

# Tests sur variables numériques (Kolmogorov-Smirnov)
num_cols = ['age', 'income', 'employment_years', 'credit_score', 
            'loan_amount', 'loan_term_months', 'interest_rate', 'debt_to_income']

for col in num_cols:
    ks_stat, p_val = stats.ks_2samp(train_df[col], test_df[col])
    homogeneity_results.append({
        'Variable': col,
        'Type': 'Numérique',
        'Test': 'Kolmogorov-Smirnov',
        'Statistique': round(ks_stat, 4),
        'p-value': round(p_val, 4),
        'Homogène': 'Oui' if p_val > 0.05 else 'Non (Biais)'
    })

# Tests sur variables catégorielles (Chi²)
homogeneity_results = []
cat_cols = ['home_ownership', 'loan_purpose', 'previous_defaults']

for col in cat_cols:
    # Tableau de contingence : lignes = Train/Test, colonnes = modalités de la variable
    train_counts = train_df[col].value_counts()
    test_counts = test_df[col].value_counts()
    obs = pd.DataFrame([train_counts, test_counts], index=['Train', 'Test']).fillna(0)
    
    # Test du Chi²
    chi2, p_val, dof, _ = stats.chi2_contingency(obs)
    
    homogeneity_results.append({
        'Variable': col,
        'Type': 'Catégorielle',
        'Test': 'Chi²',
        'Statistique': round(chi2, 4),
        'p-value': round(p_val, 4),
        'Homogène': 'Oui' if p_val > 0.05 else 'Non (Biais)'
    })

homogeneity_df = pd.DataFrame(homogeneity_results)
print("=== Bilan des Tests d'Homogénéité (Train vs Test) ===")
display(homogeneity_df)

=== Bilan des Tests d'Homogénéité (Train vs Test) ===


,Variable,Type,Test,Statistique,p-value,Homogène
0,home_ownership,Catégorielle,Chi²,0.2139,0.8986,Oui
1,loan_purpose,Catégorielle,Chi²,4.4107,0.3533,Oui
2,previous_defaults,Catégorielle,Chi²,0.8409,0.6567,Oui


**Analyse du résultat :** 

Toutes les p-values sont largement supérieures au seuil de 0.5. On valide qu'il n'y a **aucun biais d'échantillonnage** entre la cohorte d'apprentissage et la cohorte de validation. Les deux sous-ensembles sont statistiquement identiques.

## 3. Discrétisation Optimale (*Optimal Binning*) et Calcul du WoE sur Train

Pour chaque variable, nous appliquons un algorithme de découpage optimal qui respecte les contraintes prudentielles :
- **Taille minimale par classe** : au moins $5\%$ de l'effectif.
- **Monotonie du WoE** : pour les variables financières (credit_score, debt_to_income, previous_defaults), le WoE doit évoluer de façon strictement monotone pour garantir l'explicabilité de la grille de score.

**Formule du Weight of Evidence (WoE) :**
$[WoE_i = \ln \left( \frac{\% \text{Sains}_i}{\% \text{Défaillants}_i} \right)]$

*Un WoE positif indique une classe protectrice (faible risque), un WoE négatif indique une classe à risque élevé.*

In [19]:
features_to_bin = [c for c in train_df.columns if c not in ['loan_id', 'loan_status']]
target_col = 'loan_status'

binning_models = {}
iv_summary = []

for var in features_to_bin:
    is_numeric = train_df[var].dtype in ['int64', 'float64'] and train_df[var].nunique() > 5
    dtype_opt = "numerical" if is_numeric else "categorical"
    
    # Instanciation d'OptimalBinning avec contrainte de taille minimale (5%)
    optb = OptimalBinning(
        name=var, 
        dtype=dtype_opt, 
        solver="cp", 
        min_bin_size=0.05,
        monotonic_trend="auto" if is_numeric else None
    )
    
    # Fit STRICTEMENT sur le Train
    optb.fit(train_df[var].values, train_df[target_col].values)
    
    binning_models[var] = optb
    iv_summary.append({
        'Variable': var,
        'Type': dtype_opt,
        'Information_Value': round(optb.binning_table.iv, 4)
    })

iv_df = pd.DataFrame(iv_summary).sort_values(by='Information_Value', ascending=False)

def interpret_iv(iv):
    if iv < 0.02: 
        return "Inutile (< 0.02)"
    elif iv < 0.10: 
        return "Faible (0.02 - 0.10)"
    elif iv < 0.30: 
        return "Moyen / Bon (0.10 - 0.30)"
    else: 
        return "Fort (>= 0.30)"

iv_df['Pouvoir_Predictif'] = iv_df['Information_Value'].apply(interpret_iv)

print("=== Bilan des Information Values (IV) après Binning Optimal (Train) ===")
display(iv_df)


TypeError: check_array() got an unexpected keyword argument 'force_all_finite'. Did you mean 'ensure_all_finite'?